# Call-level caching — try it yourself

Cash normally caches a **statement**. When the statement is a cheap wrapper around
an expensive call, that unit is wrong:

| statement | what statement-level caching alone does | why |
|---|---|---|
| `out.append(compute(x))` | never reuses anything | the append mutates an object that already exists, so there is no snapshot to restore |
| `s += compute(x)` | reuses only an unchanged *prefix* | each iteration reads `s`, so its key encodes every iteration before it — reorder the list and the tail re-runs |

Cash fixes both of those **by default**, with no directive: it also caches the
expensive `compute(x)` call itself, one level below the statement. The cheap
wrapper still executes every run (so the append really happens); the expensive
call is served from cache. `# @cash:no-cache-calls` is the escape hatch if you
need the old, always-recompute behaviour back for a statement or a whole cell.

**How to use this notebook:** run the cells top to bottom once, then go back and
**re-run individual cells** as each section tells you. The counter and the badge
are the evidence — not the wall clock, which cannot tell "recomputed" from
"restored but slow".

In [ ]:
import cash
%cash_on

## Setup

`compute()` sleeps for a second and records that it ran. `CALLS` is the ground
truth throughout: **if a number appears in `CALLS`, that call really executed.**

> You will see a `CashImpurityWarning` about `CALLS.append()` the first time a
> cached call runs. That is cash being correct — appending to a global *is* a
> side effect, and cash is telling you a cached call will skip it. Here the side
> effect is only the counter, so it is safe to ignore; in real code it is worth
> reading.

In [ ]:
import time

CALLS = []

def compute(x):
    CALLS.append(x)      # ground truth: this line only runs on a real execution
    time.sleep(1.0)      # stand-in for real work
    return x + 1

def merge(a, b):
    return a + b

def show(label=""):
    print(f"{label:<22} CALLS = {CALLS}   ({len(CALLS)} real executions so far)")

show("after setup")

---
## 1. An append loop — cached automatically now

The list is created in **its own cell**, and that detail is load-bearing. When
`out = []` sits in the *same* cell as the loop, cash can cache the whole cell as
one unit and the problem disappears. The interesting — and much more common —
case is a container built earlier and appended to later.

Run the next cell once, then re-run **only the loop cell after it**, twice.
Watch `CALLS` — it grows by three on the FIRST run, then **stops growing**.
The append still runs every time (`out` is correctly rebuilt), but `compute(x)`
itself is served from cache. The badge still says `NOT CACHED · In-place
mutation on: out` for the *statement* — that part hasn't changed — but open the
row and you'll see the sub-call hits underneath it.

In [ ]:
out = []          # built HERE, appended to in the next cell

In [ ]:
for x in [1, 2, 3]:
    out.append(compute(x))

show("append, no directive")

### Turning it off with `# @cash:no-cache-calls`

Same shape, one comment added — this time to get the OLD behaviour back. Run
the `out2 = []` cell once, then re-run the loop cell **twice**.

Both runs add three executions. The opt-out disables call-level caching for
this statement, so `compute(x)` genuinely re-executes every time, same as
section 1 would have before this feature existed.

On the badge, the `compute()` row here carries no `[intercepted]` tag — unlike
the one in section 1 — because interception never engaged for it.

In [ ]:
out2 = []         # again, built in its own cell

In [ ]:
# @cash:no-cache-calls
for x in [1, 2, 3]:
    out2.append(compute(x))

show("append + no-cache-calls")
print("out2 =", out2)

---
## 2. Reordering a loop — free by default now

This is the one that motivated the feature.

Run the next cell once. Then **change the list to `[30, 20, 10]`** and run it
again.

The accumulator `s` still makes each iteration's *statement* depend on every
iteration before it — reordering still misses the statement-level cache, and
the loop genuinely re-executes. But `compute(x)` is cached by default now, and
a call cache keys on arguments, not execution history, so no new executions
show up: reordering costs nothing.

In [ ]:
s = 0
for x in [10, 20, 30]:      # <-- try [30, 20, 10] on the second run
    s += compute(x)

print("SUM", s)
show("fold, no directive")

### The same fold, with `# @cash:no-cache-calls`

Run once, then **reorder the list any way you like** and run again.

With interception switched off for this statement, both the statement-level
entries AND the calls miss on a reorder — genuinely every `compute(x)` in the
new order re-executes. This is the historical, pre-default-on behaviour the
docs used to describe unconditionally.

In [ ]:
s2 = 0
# @cash:no-cache-calls
for x in [11, 22, 33]:      # <-- reorder freely; every compute() call re-runs
    s2 += compute(x)

print("SUM", s2)
show("fold + no-cache-calls")

---
## 3. What is (and isn't) eligible

A call is only extractable when it does **not** read the statement's own
assignment or mutation target. If it does, it *is* the fold and there is no
order-independent value to pull out of it.

| statement | cached call |
|---|---|
| `s += compute(x)` | `compute(x)` |
| `out.append(compute(x))` | `compute(x)` |
| `prices[k] = compute(k)` | `compute(k)` |
| `s = merge(s, x)` | none — the call reads `s` |
| `df.sort_values(inplace=True)` | none — the mutation *is* the work |

The next cell is the last row of that table: `merge` reads `acc`, the statement's
own target, so there is nothing to extract. Unlike an earlier version of this
feature, cash does **not** warn about this — under default-on, "nothing here
was eligible" is the ordinary case for most statements, not a mistake. Nothing
was wrapped, so there is nothing to log: the badge's `@cash.cache` section
simply won't mention `merge()` at all, in contrast to the `compute()` rows
above it that do appear there, tagged `[intercepted]`.

In [ ]:
acc = 0
acc = merge(acc, 5)      # `merge` reads `acc`, the target -> nothing to extract; not intercepted

print("acc =", acc)

---
## What to look for

- **`CALLS`** is ground truth. A number appears only when the work really ran.
- **The badge tag** `compute() [intercepted]` confirms cash wrapped that call
  itself. Plain `@cache` (no tag) means you decorated that function yourself;
  no row at all means nothing was wrapped or decorated.
- **No warning appears when nothing is eligible.** An earlier, opt-in version
  of this feature warned in that case; under default-on it's the ordinary
  outcome for most statements, not a mistake — check the badge, not your
  warnings tab.

### What is deliberately *not* intercepted

- **Already-decorated functions.** They are on this path already; wrapping them
  again would split their hits across two cache entries.
- **Builtins.** A hot loop must not pay for a cache key per `len()`.
- **Bound methods** (`model.predict(x)`). Caching a method puts `self` in the
  key, which needs your judgement rather than cash's guess — see
  [caching class methods](https://cash-lib.readthedocs.io/en/latest/tutorials/feature-guides/caching-class-methods/).
  Decorate the method yourself when you want that.

### Reference

- [Call-level caching, and `# @cash:no-cache-calls`](https://cash-lib.readthedocs.io/en/latest/annotations/#call-level-caching-default-and-cashno-cache-calls-alias-nocachecalls)
- [Reordering a loop's items](https://cash-lib.readthedocs.io/en/latest/known-limitations/#reordering-a-loops-items-re-runs-the-tail)

> **Placement matters.** A `@cash:` directive attaches to the statement *below*
> it, and the backwards scan stops at the first non-comment line. On a loop, put
> it on the `for` header — on the cell's first line it would scope to whatever
> statement happens to be first. `no-cache-calls` (like `no-cache`) also
> propagates from a cell's very first lines to every top-level statement in
> that cell — but that's a *different* mechanism from the loop-header one, and
> it does not reach past an intervening statement between the header and a
> loop. See [the annotations reference](https://cash-lib.readthedocs.io/en/latest/annotations/#a-cell-header-opt-out-does-not-reach-past-an-intervening-statement)
> if you rely on it.